In [ ]:
"""
Insurance Claim Prediction - Model Training (Module 1: preprocessing + 4 algorithms)

Usage:
    python train_models.py --data Car_Insurance_Claim.csv
    python train_models.py --data Car_Insurance_Claim.csv --include-sensitive
    python train_models.py --data Car_Insurance_Claim.csv --n-iter 5   # quick smoke test

Outputs (in ./artifacts):
    models/<name>.joblib      fitted sklearn Pipeline (preprocessing + model)
    metrics.csv               test-set comparison of all models
    best_params.json          tuned hyper-parameters per model
    roc_curves.png, pr_curves.png, confusion_matrices.png
    test_split.csv            held-out test rows (reuse for SHAP / dashboard demo)
"""
import argparse
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     cross_val_predict, train_test_split)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
TARGET = "OUTCOME"

# ----------------------------------------------------------------------------
# Feature definitions
# ----------------------------------------------------------------------------
# Ordered categories -> ordinal encoding keeps their natural ranking.
ORDINAL = {
    "AGE": ["16-25", "26-39", "40-64", "65+"],
    "DRIVING_EXPERIENCE": ["0-9y", "10-19y", "20-29y", "30y+"],
    "EDUCATION": ["none", "high school", "university"],
    "INCOME": ["poverty", "working class", "middle class", "upper class"],
}
NOMINAL = ["VEHICLE_TYPE", "VEHICLE_YEAR", "POSTAL_CODE", "GENDER", "RACE"]
NUMERIC = ["CREDIT_SCORE", "ANNUAL_MILEAGE", "SPEEDING_VIOLATIONS", "DUIS",
           "PAST_ACCIDENTS", "VEHICLE_OWNERSHIP", "MARRIED", "CHILDREN"]
SENSITIVE = ["GENDER", "RACE"]  # protected attributes - excluded by default


def load_data(path, include_sensitive):
    df = pd.read_csv(path)
    df = df.drop(columns=["ID"])                      # identifier, no predictive meaning
    df["POSTAL_CODE"] = df["POSTAL_CODE"].astype(str)  # it is a category, not a quantity
    df[TARGET] = df[TARGET].astype(int)

    # Feature engineering (simple, domain-motivated)
    df["CREDIT_MISSING"] = df["CREDIT_SCORE"].isna().astype(int)
    df["MILEAGE_MISSING"] = df["ANNUAL_MILEAGE"].isna().astype(int)
    df["VIOLATIONS_TOTAL"] = df["SPEEDING_VIOLATIONS"] + df["DUIS" ] + df["PAST_ACCIDENTS"]
    df["HAS_PRIOR_ACCIDENT"] = (df["PAST_ACCIDENTS"] > 0).astype(int)

    ordinal = dict(ORDINAL)
    nominal = list(NOMINAL)
    numeric = NUMERIC + ["CREDIT_MISSING", "MILEAGE_MISSING", "VIOLATIONS_TOTAL", "HAS_PRIOR_ACCIDENT"]
    if not include_sensitive:
        nominal = [c for c in nominal if c not in SENSITIVE]
        df = df.drop(columns=SENSITIVE)
    return df, ordinal, nominal, numeric


def build_preprocessor(ordinal, nominal, numeric, scale):
    """Fit on training folds only (inside the Pipeline) -> no leakage."""
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    ct = ColumnTransformer(
        [
            ("ord", OrdinalEncoder(categories=list(ordinal.values())), list(ordinal.keys())),
            ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal),
            ("num", Pipeline(num_steps), numeric),
        ],
        verbose_feature_names_out=False,
    )
    ct.set_output(transform="pandas")  # keep column names for SHAP later
    return ct


# ----------------------------------------------------------------------------
# Models + search spaces
# ----------------------------------------------------------------------------
def get_models(pos_weight):
    models = {}
    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = (
            XGBClassifier(eval_metric="logloss", tree_method="hist", random_state=SEED,
                          n_jobs=-1, scale_pos_weight=pos_weight),
            {"n_estimators": randint(150, 600), "max_depth": randint(2, 7),
             "learning_rate": loguniform(0.01, 0.2), "subsample": uniform(0.6, 0.4),
             "colsample_bytree": uniform(0.6, 0.4), "min_child_weight": randint(1, 10),
             "reg_lambda": loguniform(0.1, 10)}, False)
    except ImportError:
        print("[skip] xgboost not installed")
    try:
        from lightgbm import LGBMClassifier
        models["LightGBM"] = (
            LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1,
                           scale_pos_weight=pos_weight),
            {"n_estimators": randint(150, 600), "num_leaves": randint(8, 64),
             "learning_rate": loguniform(0.01, 0.2), "subsample": uniform(0.6, 0.4),
             "colsample_bytree": uniform(0.6, 0.4), "min_child_samples": randint(10, 60),
             "reg_lambda": loguniform(0.1, 10)}, False)
    except ImportError:
        print("[skip] lightgbm not installed")
    try:
        from catboost import CatBoostClassifier
        models["CatBoost"] = (
            CatBoostClassifier(random_seed=SEED, verbose=0, allow_writing_files=False,
                               scale_pos_weight=pos_weight, thread_count=-1),
            {"iterations": randint(200, 700), "depth": randint(3, 8),
             "learning_rate": loguniform(0.01, 0.2), "l2_leaf_reg": loguniform(1, 10)}, False)
    except ImportError:
        print("[skip] catboost not installed")

    # Neural network (scikit-learn MLP; features are standardised)
    models["NeuralNetwork"] = (
        MLPClassifier(early_stopping=True, validation_fraction=0.15, n_iter_no_change=15,
                      max_iter=500, random_state=SEED),
        {"hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64), (64, 32, 16)],
         "alpha": loguniform(1e-5, 1e-1), "learning_rate_init": loguniform(1e-4, 1e-2),
         "activation": ["relu", "tanh"]}, True)
    return models


# ----------------------------------------------------------------------------
# Evaluation helpers
# ----------------------------------------------------------------------------
def best_f1_threshold(y_true, proba):
    p, r, t = precision_recall_curve(y_true, proba)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-9, None)
    return float(t[np.argmax(f1)])


def evaluate(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "brier": brier_score_loss(y_true, proba),
        "threshold": thr,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="Car_Insurance_Claim.csv")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--n-iter", type=int, default=30, help="random-search iterations per model")
    ap.add_argument("--cv", type=int, default=5)
    ap.add_argument("--include-sensitive", action="store_true",
                    help="keep GENDER and RACE as features (off by default)")

    # Use parse_known_args() so internal Jupyter notebook arguments are ignored
    args, _ = ap.parse_known_args()

    out = Path(args.out)
    (out / "models").mkdir(parents=True, exist_ok=True)

    df, ordinal, nominal, numeric = load_data(args.data, args.include_sensitive)
    X, y = df.drop(columns=[TARGET]), df[TARGET]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    print(f"Train {X_tr.shape} | Test {X_te.shape} | claim rate {y.mean():.1%} "
          f"| sensitive features {'INCLUDED' if args.include_sensitive else 'excluded'}")
    pd.concat([X_te, y_te], axis=1).to_csv(out / "test_split.csv", index=False)

    pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
    cv = StratifiedKFold(args.cv, shuffle=True, random_state=SEED)

    results, best_params, curves = [], {}, {}
    for name, (est, space, scale) in get_models(pos_weight).items():
        print(f"\n=== {name} ===")
        t0 = time.time()
        # MLP has no class_weight; the boosters use scale_pos_weight. Ranking metric
        # (ROC-AUC) is unaffected, and the threshold is tuned below.
        pipe = Pipeline([("prep", build_preprocessor(ordinal, nominal, numeric, scale)),
                         ("clf", est)])
        search = RandomizedSearchCV(
            pipe, {f"clf__{k}": v for k, v in space.items()}, n_iter=args.n_iter,
            scoring="roc_auc", cv=cv, random_state=SEED, n_jobs=1, refit=True)
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        best_params[name] = {k.replace("clf__", ""): (v.item() if hasattr(v, "item") else v)
                             for k, v in search.best_params_.items()}

        # Decision threshold chosen on out-of-fold training predictions (never on test)
        oof = cross_val_predict(model, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
        thr = best_f1_threshold(y_tr, oof)

        proba = model.predict_proba(X_te)[:, 1]
        m = evaluate(y_te, proba, thr)
        m.update(model=name, cv_roc_auc=search.best_score_, train_seconds=round(time.time() - t0, 1))
        results.append(m)
        curves[name] = (proba, thr)
        joblib.dump(model, out / "models" / f"{name}.joblib")
        print(f"CV AUC {search.best_score_:.4f} | Test AUC {m['roc_auc']:.4f} | "
              f"F1 {m['f1']:.4f} | thr {thr:.2f} | {m['train_seconds']}s")

    res = pd.DataFrame(results).set_index("model")
    cols = ["cv_roc_auc", "roc_auc", "pr_auc", "accuracy", "precision", "recall", "f1",
            "brier", "threshold", "train_seconds"]
    res = res[cols].sort_values("roc_auc", ascending=False).round(4)
    res.to_csv(out / "metrics.csv")
    (out / "best_params.json").write_text(json.dumps(best_params, indent=2, default=str))
    print("\n", res.to_string())

    # ---- plots ----
    fig, ax = plt.subplots(figsize=(6, 5))
    for n, (p, _) in curves.items():
        fpr, tpr, _ = roc_curve(y_te, p)
        ax.plot(fpr, tpr, label=f"{n} (AUC={roc_auc_score(y_te, p):.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=.4); ax.set(xlabel="FPR", ylabel="TPR", title="ROC curves (test)")
    ax.legend(); fig.tight_layout(); fig.savefig(out / "roc_curves.png", dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    for n, (p, _) in curves.items():
        pr, rc, _ = precision_recall_curve(y_te, p)
        ax.plot(rc, pr, label=f" {n} (AP={average_precision_score(y_te, p):.3f})")
    ax.axhline(y_te.mean(), color="k", ls="--", alpha=.4)
    ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (test)")
    ax.legend(); fig.tight_layout(); fig.savefig(out / "pr_curves.png", dpi=150); plt.close(fig)

    fig, axes = plt.subplots(1, len(curves), figsize=(4 * len(curves), 3.6))
    for ax, (n, (p, thr)) in zip(np.atleast_1d(axes), curves.items()):
        cm = confusion_matrix(y_te, (p >= thr).astype(int))
        ax.imshow(cm, cmap="Blues")
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, v, ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
        ax.set(title=n, xticks=[0, 1], yticks=[0, 1], xlabel="Predicted", ylabel="Actual")
    fig.tight_layout(); fig.savefig(out / "confusion_matrices.png", dpi=150); plt.close(fig)
    print(f"\nSaved everything to {out.resolve()}")


if __name__ == "__main__":
    main()

Train (8000, 19) | Test (2000, 19) | claim rate 31.3% | sensitive features excluded
[skip] catboost not installed

=== XGBoost ===
CV AUC 0.9174 | Test AUC 0.9053 | F1 0.7481 | thr 0.53 | 50.8s

=== LightGBM ===
CV AUC 0.9173 | Test AUC 0.9062 | F1 0.7514 | thr 0.57 | 70.3s

=== NeuralNetwork ===
CV AUC 0.9173 | Test AUC 0.9059 | F1 0.7386 | thr 0.32 | 253.6s

                cv_roc_auc  roc_auc  pr_auc  accuracy  precision  recall      f1   brier  threshold  train_seconds
model                                                                                                            
LightGBM           0.9173   0.9062  0.8150    0.8230     0.6713  0.8533  0.7514  0.1290     0.5744           70.3
NeuralNetwork      0.9173   0.9059  0.8142    0.8085     0.6456  0.8628  0.7386  0.1140     0.3242          253.6
XGBoost            0.9174   0.9053  0.8121    0.8205     0.6679  0.8501  0.7481  0.1281     0.5332           50.8

Saved everything to /content/artifacts


In [ ]:
"""
Accuracy report for all trained models.
Run AFTER train_models.py:   python check_accuracy.py
"""
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

ART = Path("artifacts")
test = pd.read_csv(ART / "test_split.csv")
test["POSTAL_CODE"] = test["POSTAL_CODE"].astype(str)  # models were trained on it as a category
X_test, y_test = test.drop(columns=["OUTCOME"]), test["OUTCOME"]
thresholds = pd.read_csv(ART / "metrics.csv", index_col="model")["threshold"]

rows = []
for path in sorted((ART / "models").glob("*.joblib")):
    name = path.stem
    model = joblib.load(path)
    proba = model.predict_proba(X_test)[:, 1]
    thr = thresholds[name]
    rows.append({
        "model": name,
        "accuracy @0.5": accuracy_score(y_test, proba >= 0.5),
        f"accuracy @tuned thr": accuracy_score(y_test, proba >= thr),
        "tuned thr": round(thr, 3),
    })

    print(f"\n===== {name} (threshold {thr:.2f}) =====")
    print(classification_report(y_test, (proba >= thr).astype(int),
                                target_names=["No claim", "Claim"], digits=3))

summary = pd.DataFrame(rows).set_index("model").round(4)
print("\nACCURACY SUMMARY (test set)\n", summary.sort_values("accuracy @0.5", ascending=False))
summary.to_csv(ART / "accuracy_summary.csv")


===== LightGBM (threshold 0.57) =====
              precision    recall  f1-score   support

    No claim      0.924     0.809     0.863      1373
       Claim      0.671     0.853     0.751       627

    accuracy                          0.823      2000
   macro avg      0.797     0.831     0.807      2000
weighted avg      0.844     0.823     0.828      2000


===== NeuralNetwork (threshold 0.32) =====
              precision    recall  f1-score   support

    No claim      0.926     0.784     0.849      1373
       Claim      0.646     0.863     0.739       627

    accuracy                          0.808      2000
   macro avg      0.786     0.823     0.794      2000
weighted avg      0.838     0.808     0.814      2000


===== XGBoost (threshold 0.53) =====
              precision    recall  f1-score   support

    No claim      0.922     0.807     0.861      1373
       Claim      0.668     0.850     0.748       627

    accuracy                          0.821      2000
   macr

In [ ]:
"""
Insurance Claim Prediction - Model Training (Module 1: preprocessing + 4 algorithms)

Usage:
    python train_models.py --data Car_Insurance_Claim.csv
    python train_models.py --data Car_Insurance_Claim.csv --include-sensitive
    python train_models.py --data Car_Insurance_Claim.csv --n-iter 5   # quick smoke test
    python train_models.py --data Car_Insurance_Claim.csv --class-weight  # old behaviour (distorts probabilities)

Outputs (in ./artifacts):
    models/<name>.joblib      fitted sklearn Pipeline (preprocessing + model)
    metrics.csv               test-set comparison of all models
    best_params.json          tuned hyper-parameters per model
    roc_curves.png, pr_curves.png, confusion_matrices.png, calibration_curves.png
    test_split.csv            held-out test rows (reuse for SHAP / dashboard demo)
"""
import argparse
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     cross_val_predict, train_test_split)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
TARGET = "OUTCOME"

# ----------------------------------------------------------------------------
# Feature definitions
# ----------------------------------------------------------------------------
# Ordered categories -> ordinal encoding keeps their natural ranking.
ORDINAL = {
    "AGE": ["16-25", "26-39", "40-64", "65+"],
    "DRIVING_EXPERIENCE": ["0-9y", "10-19y", "20-29y", "30y+"],
    "EDUCATION": ["none", "high school", "university"],
    "INCOME": ["poverty", "working class", "middle class", "upper class"],
}
NOMINAL = ["VEHICLE_TYPE", "VEHICLE_YEAR", "POSTAL_CODE", "GENDER", "RACE"]
NUMERIC = ["CREDIT_SCORE", "ANNUAL_MILEAGE", "SPEEDING_VIOLATIONS", "DUIS",
           "PAST_ACCIDENTS", "VEHICLE_OWNERSHIP", "MARRIED", "CHILDREN"]
SENSITIVE = ["GENDER", "RACE"]  # protected attributes - excluded by default


def load_data(path, include_sensitive):
    df = pd.read_csv(path)
    df = df.drop(columns=["ID"])                      # identifier, no predictive meaning
    df["POSTAL_CODE"] = df["POSTAL_CODE"].astype(str)  # it is a category, not a quantity
    df[TARGET] = df[TARGET].astype(int)

    # Feature engineering (simple, domain-motivated)
    df["CREDIT_MISSING"] = df["CREDIT_SCORE"].isna().astype(int)
    df["MILEAGE_MISSING"] = df["ANNUAL_MILEAGE"].isna().astype(int)
    df["VIOLATIONS_TOTAL"] = df["SPEEDING_VIOLATIONS"] + df["DUIS"] + df["PAST_ACCIDENTS"]
    df["HAS_PRIOR_ACCIDENT"] = (df["PAST_ACCIDENTS"] > 0).astype(int)

    ordinal = dict(ORDINAL)
    nominal = list(NOMINAL)
    numeric = NUMERIC + ["CREDIT_MISSING", "MILEAGE_MISSING", "VIOLATIONS_TOTAL", "HAS_PRIOR_ACCIDENT"]
    if not include_sensitive:
        nominal = [c for c in nominal if c not in SENSITIVE]
        df = df.drop(columns=SENSITIVE)
    return df, ordinal, nominal, numeric


def build_preprocessor(ordinal, nominal, numeric, scale):
    """Fit on training folds only (inside the Pipeline) -> no leakage."""
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    ct = ColumnTransformer(
        [
            ("ord", OrdinalEncoder(categories=list(ordinal.values())), list(ordinal.keys())),
            ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal),
            ("num", Pipeline(num_steps), numeric),
        ],
        verbose_feature_names_out=False,
    )
    ct.set_output(transform="pandas")  # keep column names for SHAP later
    return ct


# ----------------------------------------------------------------------------
# Models + search spaces
# ----------------------------------------------------------------------------
def get_models(pos_weight):
    models = {}
    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = (
            XGBClassifier(eval_metric="logloss", tree_method="hist", random_state=SEED,
                          n_jobs=-1, scale_pos_weight=pos_weight),
            {"n_estimators": randint(150, 600), "max_depth": randint(2, 7),
             "learning_rate": loguniform(0.01, 0.2), "subsample": uniform(0.6, 0.4),
             "colsample_bytree": uniform(0.6, 0.4), "min_child_weight": randint(1, 10),
             "reg_lambda": loguniform(0.1, 10)}, False)
    except ImportError:
        print("[skip] xgboost not installed")
    try:
        from lightgbm import LGBMClassifier
        models["LightGBM"] = (
            LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1,
                           scale_pos_weight=pos_weight),
            {"n_estimators": randint(150, 600), "num_leaves": randint(8, 64),
             "learning_rate": loguniform(0.01, 0.2), "subsample": uniform(0.6, 0.4),
             "colsample_bytree": uniform(0.6, 0.4), "min_child_samples": randint(10, 60),
             "reg_lambda": loguniform(0.1, 10)}, False)
    except ImportError:
        print("[skip] lightgbm not installed")
    try:
        from catboost import CatBoostClassifier
        models["CatBoost"] = (
            CatBoostClassifier(random_seed=SEED, verbose=0, allow_writing_files=False,
                               scale_pos_weight=pos_weight, thread_count=-1),
            {"iterations": randint(200, 700), "depth": randint(3, 8),
             "learning_rate": loguniform(0.01, 0.2), "l2_leaf_reg": loguniform(1, 10)}, False)
    except ImportError:
        print("[skip] catboost not installed")

    # Neural network (scikit-learn MLP; features are standardised)
    models["NeuralNetwork"] = (
        MLPClassifier(early_stopping=True, validation_fraction=0.15, n_iter_no_change=15,
                      max_iter=500, random_state=SEED),
        {"hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64), (64, 32, 16)],
         "alpha": loguniform(1e-5, 1e-1), "learning_rate_init": loguniform(1e-4, 1e-2),
         "activation": ["relu", "tanh"]}, True)
    return models


# ----------------------------------------------------------------------------
# Evaluation helpers
# ----------------------------------------------------------------------------
def best_f1_threshold(y_true, proba):
    p, r, t = precision_recall_curve(y_true, proba)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-9, None)
    return float(t[np.argmax(f1)])


def expected_calibration_error(y_true, proba, bins=10):
    """Average gap between predicted probability and observed claim rate (lower = better)."""
    y_true = np.asarray(y_true)
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(proba, edges) - 1, 0, bins - 1)
    ece = 0.0
    for b in range(bins):
        mask = idx == b
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - proba[mask].mean())
    return float(ece)


def evaluate(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "brier": brier_score_loss(y_true, proba),
        "ece": expected_calibration_error(y_true, proba),
        "threshold": thr,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="Car_Insurance_Claim.csv")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--n-iter", type=int, default=30, help="random-search iterations per model")
    ap.add_argument("--cv", type=int, default=5)
    ap.add_argument("--class-weight", action="store_true",
                    help="up-weight claims in boosters (raises recall but distorts probabilities)")
    ap.add_argument("--include-sensitive", action="store_true",
                    help="keep GENDER and RACE as features (off by default)")

    # Use parse_known_args() so internal Jupyter notebook arguments are ignored
    args, _ = ap.parse_known_args()

    out = Path(args.out)
    (out / "models").mkdir(parents=True, exist_ok=True)

    df, ordinal, nominal, numeric = load_data(args.data, args.include_sensitive)
    X, y = df.drop(columns=[TARGET]), df[TARGET]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    print(f"Train {X_tr.shape} | Test {X_te.shape} | claim rate {y.mean():.1%} "
          f"| sensitive features {'INCLUDED' if args.include_sensitive else 'excluded'}")
    pd.concat([X_te, y_te], axis=1).to_csv(out / "test_split.csv", index=False)

    # Default 1.0 = no re-weighting -> probabilities stay honest (needed for the risk score).
    # Class imbalance is handled by the tuned decision threshold instead.
    pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum() if args.class_weight else 1.0
    print(f"scale_pos_weight = {pos_weight:.2f}")
    cv = StratifiedKFold(args.cv, shuffle=True, random_state=SEED)

    results, best_params, curves = [], {}, {}
    for name, (est, space, scale) in get_models(pos_weight).items():
        print(f"\n=== {name} ===")
        t0 = time.time()
        # MLP has no class_weight; the boosters use scale_pos_weight. Ranking metric
        # (ROC-AUC) is unaffected, and the threshold is tuned below.
        pipe = Pipeline([("prep", build_preprocessor(ordinal, nominal, numeric, scale)),
                         ("clf", est)])
        search = RandomizedSearchCV(
            pipe, {f"clf__{k}": v for k, v in space.items()}, n_iter=args.n_iter,
            scoring="roc_auc", cv=cv, random_state=SEED, n_jobs=1, refit=True)
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        best_params[name] = {k.replace("clf__", ""): (v.item() if hasattr(v, "item") else v)
                             for k, v in search.best_params_.items()}

        # Decision threshold chosen on out-of-fold training predictions (never on test)
        oof = cross_val_predict(model, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
        thr = best_f1_threshold(y_tr, oof)

        proba = model.predict_proba(X_te)[:, 1]
        m = evaluate(y_te, proba, thr)
        m.update(model=name, cv_roc_auc=search.best_score_, train_seconds=round(time.time() - t0, 1))
        results.append(m)
        curves[name] = (proba, thr)
        joblib.dump(model, out / "models" / f"{name}.joblib")
        print(f"CV AUC {search.best_score_:.4f} | Test AUC {m['roc_auc']:.4f} | "
              f"F1 {m['f1']:.4f} | thr {thr:.2f} | {m['train_seconds']}s")

    res = pd.DataFrame(results).set_index("model")
    cols = ["cv_roc_auc", "roc_auc", "pr_auc", "accuracy", "precision", "recall", "f1",
            "brier", "ece", "threshold", "train_seconds"]
    res = res[cols].sort_values("roc_auc", ascending=False).round(4)
    res.to_csv(out / "metrics.csv")
    (out / "best_params.json").write_text(json.dumps(best_params, indent=2, default=str))
    print("\n", res.to_string())

    # ---- plots ----
    fig, ax = plt.subplots(figsize=(6, 5))
    for n, (p, _) in curves.items():
        fpr, tpr, _ = roc_curve(y_te, p)
        ax.plot(fpr, tpr, label=f"{n} (AUC={roc_auc_score(y_te, p):.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=.4); ax.set(xlabel="FPR", ylabel="TPR", title="ROC curves (test)")
    ax.legend(); fig.tight_layout(); fig.savefig(out / "roc_curves.png", dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    for n, (p, _) in curves.items():
        pr, rc, _ = precision_recall_curve(y_te, p)
        ax.plot(rc, pr, label=f"{n} (AP={average_precision_score(y_te, p):.3f})")
    ax.axhline(y_te.mean(), color="k", ls="--", alpha=.4)
    ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (test)")
    ax.legend(); fig.tight_layout(); fig.savefig(out / "pr_curves.png", dpi=150); plt.close(fig)

    fig, axes = plt.subplots(1, len(curves), figsize=(4 * len(curves), 3.6))
    for ax, (n, (p, thr)) in zip(np.atleast_1d(axes), curves.items()):
        cm = confusion_matrix(y_te, (p >= thr).astype(int))
        ax.imshow(cm, cmap="Blues")
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, v, ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
        ax.set(title=n, xticks=[0, 1], yticks=[0, 1], xlabel="Predicted", ylabel="Actual")
    fig.tight_layout(); fig.savefig(out / "confusion_matrices.png", dpi=150); plt.close(fig)
    fig, ax = plt.subplots(figsize=(6, 5))
    for n, (p, _) in curves.items():
        frac, mean_pred = calibration_curve(y_te, p, n_bins=10, strategy="quantile")
        ax.plot(mean_pred, frac, marker="o", label=n)
    ax.plot([0, 1], [0, 1], "k--", alpha=.4, label="Perfect")
    ax.set(xlabel="Predicted probability", ylabel="Observed claim rate", title="Calibration (test)")
    ax.legend(); fig.tight_layout(); fig.savefig(out / "calibration_curves.png", dpi=150); plt.close(fig)

    print(f"\nSaved everything to {out.resolve()}")


if __name__ == "__main__":
    main()

Train (8000, 19) | Test (2000, 19) | claim rate 31.3% | sensitive features excluded
scale_pos_weight = 1.00
[skip] catboost not installed

=== XGBoost ===
CV AUC 0.9176 | Test AUC 0.9057 | F1 0.7514 | thr 0.35 | 50.5s

=== LightGBM ===
CV AUC 0.9174 | Test AUC 0.9058 | F1 0.7493 | thr 0.36 | 68.7s

=== NeuralNetwork ===
CV AUC 0.9173 | Test AUC 0.9059 | F1 0.7386 | thr 0.32 | 232.3s

                cv_roc_auc  roc_auc  pr_auc  accuracy  precision  recall      f1   brier     ece  threshold  train_seconds
model                                                                                                                    
NeuralNetwork      0.9173   0.9059  0.8142    0.8085     0.6456  0.8628  0.7386  0.1140  0.0162     0.3242          232.3
LightGBM           0.9174   0.9058  0.8136    0.8200     0.6650  0.8581  0.7493  0.1139  0.0251     0.3586           68.7
XGBoost            0.9176   0.9057  0.8143    0.8220     0.6683  0.8581  0.7514  0.1136  0.0190     0.3549           50.5

S

In [ ]:
"""
Module 2 & 3: Exploratory Data Analysis + Customer Risk Profiling

Usage:
    python eda_risk_profiling.py --data Car_Insurance_Claim.csv --model XGBoost

Outputs (in ./artifacts/eda and ./artifacts/risk):
    eda/*.png                      EDA figures
    eda/claim_rate_by_feature.csv  claim rate for every category
    risk/risk_tier_summary.csv     size, avg score and ACTUAL claim rate per tier
    risk/tier_profiles.csv         average customer characteristics per tier
    risk/risk_score_distribution.png, risk/tier_claim_rate.png
    risk/risk_config.json          cut-offs used by the dashboard
"""
import argparse
import json
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TARGET = "OUTCOME"
CATEGORICAL = ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME",
               "VEHICLE_OWNERSHIP", "VEHICLE_YEAR", "MARRIED", "CHILDREN", "POSTAL_CODE",
               "VEHICLE_TYPE", "DUIS", "PAST_ACCIDENTS", "SPEEDING_VIOLATIONS"]
ORDER = {"AGE": ["16-25", "26-39", "40-64", "65+"],
         "DRIVING_EXPERIENCE": ["0-9y", "10-19y", "20-29y", "30y+"],
         "EDUCATION": ["none", "high school", "university"],
         "INCOME": ["poverty", "working class", "middle class", "upper class"]}

# Risk tiers on the calibrated claim probability (score = 100 * probability)
TIER_CUTOFFS = {"Low": (0.0, 0.20), "Medium": (0.20, 0.50), "High": (0.50, 1.01)}


def risk_score(proba):
    """Calibrated claim probability -> 0-100 risk score."""
    return np.round(np.asarray(proba) * 100, 1)


def risk_tier(proba):
    """Map probability -> 'Low' / 'Medium' / 'High' (also used by the dashboard)."""
    p = np.asarray(proba)
    return np.select([p < 0.20, p < 0.50], ["Low", "Medium"], default="High")


# ---------------------------------------------------------------------------
def run_eda(df, out):
    out.mkdir(parents=True, exist_ok=True)

    # 1. target balance
    fig, ax = plt.subplots(figsize=(4, 4))
    vc = df[TARGET].value_counts().sort_index()
    ax.bar(["No claim", "Claim"], vc.values, color=["#4C78A8", "#E45756"])
    for i, v in enumerate(vc.values):
        ax.text(i, v, f"{v}\n({v / len(df):.1%})", ha="center", va="bottom")
    ax.set(title="Target distribution", ylim=(0, vc.max() * 1.18))
    fig.tight_layout(); fig.savefig(out / "target_distribution.png", dpi=150); plt.close(fig)

    # 2. missing values
    miss = df.isna().mean().mul(100)
    miss = miss[miss > 0]
    if len(miss):
        fig, ax = plt.subplots(figsize=(5, 3))
        miss.plot.barh(ax=ax, color="#F58518"); ax.set(xlabel="% missing", title="Missing values")
        fig.tight_layout(); fig.savefig(out / "missing_values.png", dpi=150); plt.close(fig)

    # 3. claim rate by category (also saved as a table)
    rows = []
    for c in CATEGORICAL:
        g = df.groupby(c)[TARGET].agg(["mean", "count"]).reset_index()
        if c in ORDER:
            g[c] = pd.Categorical(g[c], ORDER[c], ordered=True); g = g.sort_values(c)
        for _, r in g.iterrows():
            rows.append({"feature": c, "value": r[c], "claim_rate": round(r["mean"], 4),
                         "customers": int(r["count"])})
    pd.DataFrame(rows).to_csv(out / "claim_rate_by_feature.csv", index=False)

    cols = 4
    nrows = int(np.ceil(len(CATEGORICAL) / cols))
    fig, axes = plt.subplots(nrows, cols, figsize=(4 * cols, 3 * nrows))
    for ax, c in zip(axes.ravel(), CATEGORICAL):
        g = df.groupby(c)[TARGET].mean()
        if c in ORDER: g = g.reindex(ORDER[c])
        g.plot.bar(ax=ax, color="#4C78A8"); ax.axhline(df[TARGET].mean(), color="r", ls="--", lw=1)
        ax.set(title=c, xlabel="", ylabel="claim rate", ylim=(0, 1)); ax.tick_params(axis="x", rotation=45)
    for ax in axes.ravel()[len(CATEGORICAL):]: ax.axis("off")
    fig.suptitle("Claim rate by feature (red line = overall rate)", y=1.0)
    fig.tight_layout(); fig.savefig(out / "claim_rate_by_feature.png", dpi=130); plt.close(fig)

    # 4. numeric distributions split by outcome
    nums = ["CREDIT_SCORE", "ANNUAL_MILEAGE"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, c in zip(axes, nums):
        for val, lab, col in [(0, "No claim", "#4C78A8"), (1, "Claim", "#E45756")]:
            ax.hist(df.loc[df[TARGET] == val, c].dropna(), bins=30, alpha=.6, label=lab, color=col)
        ax.set(title=c); ax.legend()
    fig.tight_layout(); fig.savefig(out / "numeric_distributions.png", dpi=150); plt.close(fig)

    # 5. correlation with target (ordinal features encoded by rank)
    enc = df.copy()
    for c, o in ORDER.items(): enc[c] = enc[c].map({v: i for i, v in enumerate(o)})
    enc = enc.drop(columns=["ID"], errors="ignore").select_dtypes("number")
    corr = enc.corr()
    fig, ax = plt.subplots(figsize=(9, 7.5))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
    fig.colorbar(im, shrink=.8); ax.set_title("Correlation matrix")
    fig.tight_layout(); fig.savefig(out / "correlation_heatmap.png", dpi=150); plt.close(fig)

    t = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
    print("\nTop correlations with OUTCOME:\n", t.round(3).head(8).to_string())


# ---------------------------------------------------------------------------
def run_risk_profiling(df, model_path, art, out):
    out.mkdir(parents=True, exist_ok=True)
    model = joblib.load(model_path)
    # Score the held-out test set only: training rows would give optimistic scores
    test = pd.read_csv(art / "test_split.csv")
    test["POSTAL_CODE"] = test["POSTAL_CODE"].astype(str)
    X, y = test.drop(columns=[TARGET]), test[TARGET]
    proba = model.predict_proba(X)[:, 1]

    test["risk_score"] = risk_score(proba)
    test["risk_tier"] = risk_tier(proba)
    order = ["Low", "Medium", "High"]

    summ = (test.groupby("risk_tier")
            .agg(customers=(TARGET, "size"), avg_risk_score=("risk_score", "mean"),
                 actual_claim_rate=(TARGET, "mean"))
            .reindex(order).round(3))
    summ["share_of_customers"] = (summ["customers" ] / len(test)).round(3)
    summ.to_csv(out / "risk_tier_summary.csv")
    print("\nRisk tier summary (test set):\n", summ.to_string())

    # what a typical customer in each tier looks like
    prof = test.groupby("risk_tier").agg(
        avg_past_accidents=("PAST_ACCIDENTS", "mean"), avg_speeding=("SPEEDING_VIOLATIONS", "mean"),
        avg_duis=("DUIS", "mean"), avg_credit=("CREDIT_SCORE", "mean"),
        avg_mileage=("ANNUAL_MILEAGE", "mean"),
        pct_novice_0_9y=("DRIVING_EXPERIENCE", lambda s: (s == "0-9y").mean()),
        pct_age_16_25=("AGE", lambda s: (s == "16-25").mean()),
        pct_sports_car=("VEHICLE_TYPE", lambda s: (s == "sports car").mean()),
    ).reindex(order).round(3)
    prof.to_csv(out / "tier_profiles.csv")
    print("\nTier profiles:\n", prof.T.to_string())

    # figures
    colors = {"Low": "#54A24B", "Medium": "#F2CF5B", "High": "#E45756"}
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.hist(test["risk_score"], bins=40, color="#4C78A8")
    for cut in (20, 50): ax.axvline(cut, color="k", ls="--", lw=1)
    ax.set(xlabel="Risk score (0-100)", ylabel="Customers", title="Risk score distribution")
    fig.tight_layout(); fig.savefig(out / "risk_score_distribution.png", dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 3.8))
    ax.bar(order, summ["actual_claim_rate"], color=[colors[t] for t in order])
    for i, v in enumerate(summ["actual_claim_rate"]): ax.text(i, v, f"{v:.0%}", ha="center", va="bottom")
    ax.set(ylabel="Actual claim rate", ylim=(0, 1.05), title="Do tiers separate real risk?")
    fig.tight_layout(); fig.savefig(out / "tier_claim_rate.png", dpi=150); plt.close(fig)

    (out / "risk_config.json").write_text(json.dumps(
        {"score": "100 * calibrated claim probability", "tiers": TIER_CUTOFFS,
         "model": Path(model_path).stem}, indent=2))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="Car_Insurance_Claim.csv")
    ap.add_argument("--art", default="artifacts")
    ap.add_argument("--model", default="XGBoost")

    # Use parse_known_args() so internal Jupyter notebook arguments are ignored
    a, _ = ap.parse_known_args()
    art = Path(a.art)

    df = pd.read_csv(a.data)
    df["POSTAL_CODE"] = df["POSTAL_CODE"].astype(str)
    run_eda(df, art / "eda")
    run_risk_profiling(df, art / "models" / f"{a.model}.joblib", art, art / "risk")
    print(f"\nDone. See {art.resolve()}/eda and /risk")


if __name__ == "__main__":
    main()


Top correlations with OUTCOME:
 DRIVING_EXPERIENCE    -0.497
AGE                   -0.448
INCOME                -0.423
VEHICLE_OWNERSHIP     -0.379
CREDIT_SCORE          -0.325
PAST_ACCIDENTS        -0.311
SPEEDING_VIOLATIONS   -0.292
MARRIED               -0.262

Risk tier summary (test set):
            customers  avg_risk_score  actual_claim_rate  share_of_customers
risk_tier                                                                  
Low             1069        5.222000              0.052               0.534
Medium           260       35.323002              0.338               0.130
High             671       74.466003              0.720               0.336

Tier profiles:
 risk_tier                 Low     Medium       High
avg_past_accidents      1.774      0.485      0.134
avg_speeding            2.459      1.073      0.246
avg_duis                0.368      0.177      0.031
avg_credit              0.570      0.533      0.433
avg_mileage         11140.530  11375.546  1251

In [ ]:
"""
Module 5: Explainable AI (SHAP + LIME)

Usage:
    python explain_model.py --model XGBoost
    python explain_model.py --model LightGBM --art artifacts

Outputs (in ./artifacts/explain):
    shap_importance.csv        mean |SHAP| per feature (one-hot columns merged back)
    shap_bar.png               global importance
    shap_beeswarm.png          direction + magnitude of every feature
    shap_dependence_*.png      how the top features change the prediction
    shap_waterfall_<tier>.png  why three example customers got their score
    lime_<tier>.png            LIME explanation for the same customers
    local_explanations.csv     top factors per example customer (SHAP vs LIME side by side)

The ClaimExplainer class is reused by the dashboard (explain_row).
"""
import argparse
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TARGET = "OUTCOME"
NOMINAL_PREFIXES = ["VEHICLE_TYPE", "VEHICLE_YEAR", "POSTAL_CODE", "GENDER", "RACE"]
TREE_MODELS = ("XGB", "LGBM", "CatBoost")


def group_name(col):
    """'POSTAL_CODE_10238' -> 'POSTAL_CODE'; everything else unchanged."""
    for p in NOMINAL_PREFIXES:
        if col.startswith(p + "_"):
            return p
    return col


def group_shap(sv, columns):
    """Sum SHAP values of one-hot columns back into their original feature (SHAP is additive)."""
    sv = pd.DataFrame(sv, columns=columns)
    return sv.T.groupby(group_name).sum().T


class ClaimExplainer:
    """Wraps a fitted Pipeline(prep -> clf) and produces SHAP explanations."""

    def __init__(self, pipeline, background=None):
        import shap
        self.shap = shap
        self.prep, self.clf = pipeline.named_steps["prep"], pipeline.named_steps["clf"]
        self.is_tree = type(self.clf).__name__.startswith(TREE_MODELS)
        if self.is_tree:
            self.explainer = shap.TreeExplainer(self.clf)
            self.units = "log-odds"
        else:  # e.g. neural network -> slow, model-agnostic, works in probability space
            if background is None:
                raise ValueError("Non-tree models need a background sample of raw rows")
            bg = shap.sample(self.prep.transform(background), 50, random_state=0)
            self.explainer = shap.KernelExplainer(lambda d: self.clf.predict_proba(d)[:, 1], bg)
            self.units = "probability"

    def transform(self, X_raw):
        return self.prep.transform(X_raw)

    def shap_values(self, Xt, nsamples=200):
        """Returns (n x features array for the 'claim' class, base value)."""
        if self.is_tree:
            sv = self.explainer.shap_values(Xt)
        else:
            sv = self.explainer.shap_values(Xt, nsamples=nsamples)
        if isinstance(sv, list):          # older SHAP: one array per class
            sv = sv[1]
        sv = np.asarray(sv)
        if sv.ndim == 3:                  # newer SHAP: (n, features, classes)
            sv = sv[:, :, 1]
        base = np.atleast_1d(self.explainer.expected_value)
        base = float(base[1] if len(base) > 1 else base[0])
        return sv, base

    def explain_row(self, row_raw, top_n=6):
        """Top factors for ONE customer (DataFrame with one row of raw features)."""
        Xt = self.transform(row_raw)
        sv, base = self.shap_values(Xt)
        g = group_shap(sv, Xt.columns).iloc[0]
        top = g.reindex(g.abs().sort_values(ascending=False).index).head(top_n)
        return pd.DataFrame({
            "feature": top.index,
            "value": [row_raw.iloc[0][f] if f in row_raw.columns else "" for f in top.index],
            "impact": top.values.round(4),
            "effect": np.where(top.values > 0, "raises risk", "lowers risk"),
        }), base


# ----------------------------------------------------------------------------
# LIME helpers: LIME needs a numeric array, so categoricals become integer codes
# ----------------------------------------------------------------------------
def build_lime(pipeline, X_ref):
    from lime.lime_tabular import LimeTabularExplainer
    X_ref = X_ref.copy()
    cat_cols = [c for c in X_ref.columns if X_ref[c].dtype == object or str(X_ref[c].dtype) in ("str", "string")]
    cats = {c: sorted(X_ref[c].astype(str).unique()) for c in cat_cols}
    medians = X_ref.drop(columns=cat_cols).median()

    def encode(df):
        out = df.copy()
        for c in cat_cols:
            out[c] = out[c].astype(str).map({v: i for i, v in enumerate(cats[c])})
        for c in medians.index:
            out[c] = out[c].fillna(medians[c])      # LIME cannot handle NaN
        return out.astype(float).values

    def decode(arr):
        df = pd.DataFrame(arr, columns=X_ref.columns)
        for c in cat_cols:
            df[c] = df[c].round().astype(int).clip(0, len(cats[c]) - 1).map(dict(enumerate(cats[c])))
        return df

    def predict_fn(arr):
        return pipeline.predict_proba(decode(arr))

    explainer = LimeTabularExplainer(
        encode(X_ref), feature_names=list(X_ref.columns),
        categorical_features=[list(X_ref.columns).index(c) for c in cat_cols],
        categorical_names={list(X_ref.columns).index(c): cats[c] for c in cat_cols},
        class_names=["No claim", "Claim"], mode="classification", random_state=42)
    return explainer, encode, predict_fn


# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--art", default="artifacts")
    ap.add_argument("--model", default="XGBoost")
    ap.add_argument("--max-rows", type=int, default=2000, help="rows used for global SHAP")

    # Use parse_known_args() so internal Jupyter notebook arguments are ignored
    a, _ = ap.parse_known_args()

    import shap
    art = Path(a.art)
    out = art / "explain"
    out.mkdir(parents=True, exist_ok=True)

    pipe = joblib.load(art / "models" / f"{a.model}.joblib")
    test = pd.read_csv(art / "test_split.csv")
    test["POSTAL_CODE"] = test["POSTAL_CODE"].astype(str)
    X, y = test.drop(columns=[TARGET]), test[TARGET]
    X = X.iloc[: a.max_rows if pipe.named_steps["clf"].__class__.__name__.startswith(TREE_MODELS) else 200]
    y = y.loc[X.index]

    ex = ClaimExplainer(pipe, background=X)
    Xt = ex.transform(X)
    sv, base = ex.shap_values(Xt)
    proba = pipe.predict_proba(X)[:, 1]
    print(f"SHAP computed for {len(X)} customers ({ex.units}); base value = {base:.3f}")

    if ex.is_tree:  # additivity sanity check: base + sum(SHAP) must equal the model's log-odds
        recon = base + sv.sum(axis=1)
        logit = np.log(np.clip(proba, 1e-9, 1 - 1e-9) / (1 - np.clip(proba, 1e-9, 1 - 1e-9)))
        print(f"Additivity check, max |error|: {np.abs(recon - logit).max():.4f}")

    # ---- global importance (grouped) ----
    grouped = group_shap(sv, Xt.columns)
    imp = grouped.abs().mean().sort_values(ascending=False)
    imp_df = pd.DataFrame({"feature": imp.index, "mean_abs_shap": imp.values.round(4),
                           "share_%": (imp.values / imp.sum() * 100).round(1)})
    imp_df.to_csv(out / "shap_importance.csv", index=False)
    print("\nTop features:\n", imp_df.head(10).to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 6))
    top = imp.head(15)[::-1]
    ax.barh(top.index, top.values, color="#4C78A8")
    ax.set(xlabel=f"mean |SHAP| ({ex.units})", title=f"Global feature importance - {a.model}")
    fig.tight_layout(); fig.savefig(out / "shap_bar.png", dpi=150); plt.close(fig)

    shap.summary_plot(sv, Xt, show=False, max_display=15)
    plt.title(f"SHAP summary - {a.model}")
    plt.tight_layout(); plt.savefig(out / "shap_beeswarm.png", dpi=150, bbox_inches="tight"); plt.close()

    for f in [c for c in Xt.columns if group_name(c) == c and c in imp.index][:4]:
        shap.dependence_plot(f, sv, Xt, show=False)
        plt.tight_layout(); plt.savefig(out / f"shap_dependence_{f}.png", dpi=150, bbox_inches="tight"); plt.close()

    # ---- three example customers: low / medium / high predicted risk ----
    targets = {"low": 0.05, "medium": 0.40, "high": 0.85}
    lime_exp, encode, predict_fn = build_lime(pipe, X)
    rows = []
    for tier, tp in targets.items():
        i = int(np.argmin(np.abs(proba - tp)))
        exp = shap.Explanation(values=sv[i], base_values=base, data=Xt.iloc[i].values,
                               feature_names=list(Xt.columns))
        shap.plots.waterfall(exp, show=False, max_display=10)
        plt.title(f"{tier.capitalize()}-risk customer: score {proba[i] * 100:.0f}/100 "
                  f"(actual: {'claim' if y.iloc[i] else 'no claim'})")
        plt.savefig(out / f"shap_waterfall_{tier}.png", dpi=150, bbox_inches="tight"); plt.close()

        l = lime_exp.explain_instance(encode(X.iloc[[i]])[0], predict_fn, num_features=8, num_samples=3000)
        fig = l.as_pyplot_figure(); fig.suptitle(f"LIME - {tier}-risk customer", y=1.0)
        fig.tight_layout(); fig.savefig(out / f"lime_{tier}.png", dpi=150); plt.close(fig)

        s_top, _ = ex.explain_row(X.iloc[[i]], top_n=6)
        for k, r in s_top.iterrows():
            rows.append({"customer_type": tier, "risk_score": round(proba[i] * 100, 1), "method": "SHAP",
                         "feature": r["feature"], "value": r["value"], "impact": r["impact"], "effect": r["effect"]})
        for rule, w in l.as_list():
            rows.append({"customer_type": tier, "risk_score": round(proba[i] * 100, 1), "method": "LIME",
                         "feature": rule, "value": "", "impact": round(w, 4),
                         "effect": "raises risk" if w > 0 else "lowers risk"})
    pd.DataFrame(rows).to_csv(out / "local_explanations.csv", index=False)
    print(f"\nSaved explanations to {out.resolve()}")


if __name__ == "__main__":
    main()

SHAP computed for 2000 customers (log-odds); base value = -0.795
Additivity check, max |error|: 0.0000

Top features:
            feature  mean_abs_shap   share_%
DRIVING_EXPERIENCE         1.3165 33.200001
 VEHICLE_OWNERSHIP         0.6754 17.000000
       POSTAL_CODE         0.5777 14.600000
      VEHICLE_YEAR         0.5612 14.200000
    ANNUAL_MILEAGE         0.2071  5.200000
  VIOLATIONS_TOTAL         0.1346  3.400000
           MARRIED         0.1254  3.200000
      CREDIT_SCORE         0.1024  2.600000
          CHILDREN         0.0631  1.600000
               AGE         0.0551  1.400000

Saved explanations to /content/artifacts/explain


In [ ]:
!pip freeze | grep -i -E "scikit-learn|xgboost|lightgbm|catboost|pandas|numpy|shap|joblib" > versions.txt
!zip -r artifacts.zip artifacts versions.txt train_models.py claim_service.py eda_risk_profiling.py explain_model.py requirements.txt
from google.colab import files
files.download("artifacts.zip")

	zip warning: name not matched: train_models.py
	zip warning: name not matched: claim_service.py
	zip warning: name not matched: eda_risk_profiling.py
	zip warning: name not matched: explain_model.py
	zip warning: name not matched: requirements.txt
  adding: artifacts/ (stored 0%)
  adding: artifacts/metrics.csv (deflated 38%)
  adding: artifacts/risk/ (stored 0%)
  adding: artifacts/risk/tier_claim_rate.png (deflated 18%)
  adding: artifacts/risk/tier_profiles.csv (deflated 34%)
  adding: artifacts/risk/risk_config.json (deflated 36%)
  adding: artifacts/risk/risk_score_distribution.png (deflated 20%)
  adding: artifacts/risk/risk_tier_summary.csv (deflated 19%)
  adding: artifacts/calibration_curves.png (deflated 7%)
  adding: artifacts/eda/ (stored 0%)
  adding: artifacts/eda/numeric_distributions.png (deflated 20%)
  adding: artifacts/eda/target_distribution.png (deflated 20%)
  adding: artifacts/eda/correlation_heatmap.png (deflated 19%)
  adding: artifacts/eda/claim_rate_by_featu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>